In [1]:
!pip install torch torchaudio transformers numpy scipy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
DATA_ROOT = "/kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
EPOCHS = 20

In [4]:
!pip install pykan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 2.5 MB/s eta 0:00:00


In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchaudio

from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
from sklearn.metrics import roc_curve

In [24]:
DATA_ROOT = "/kaggle/input/asvpoof-2019-dataset"

# Inside this, there is LA/LA/
LA_ROOT = f"{DATA_ROOT}/LA/LA"

TRAIN_DIR = f"{LA_ROOT}/ASVspoof2019_LA_train"
DEV_DIR   = f"{LA_ROOT}/ASVspoof2019_LA_dev"

PROTOCOL_DIR = f"{LA_ROOT}/ASVspoof2019_LA_cm_protocols"

TRAIN_PROTOCOL = f"{PROTOCOL_DIR}/ASVspoof2019.LA.cm.train.trn.txt"
DEV_PROTOCOL   = f"{PROTOCOL_DIR}/ASVspoof2019.LA.cm.dev.trl.txt"

BATCH_SIZE = 4
EPOCHS = 20
SR = 16000

In [8]:
def compute_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    return eer

In [9]:
def collate_fn(batch):
    audios, cqccs, labels = zip(*batch)

    audios = torch.nn.utils.rnn.pad_sequence(
        audios, batch_first=True
    )

    cqccs = torch.stack(cqccs)
    labels = torch.tensor(labels)

    return audios, cqccs, labels

In [10]:
class ASVspoof2019Dataset(Dataset):
    def __init__(self, base_dir, protocol_file):
        self.base_dir = base_dir
        self.samples = []

        with open(protocol_file) as f:
            for line in f:
                parts = line.strip().split()
                utt_id = parts[1]
                label = 1 if parts[3] == "spoof" else 0
                self.samples.append((utt_id, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        utt_id, label = self.samples[idx]

        audio_path = os.path.join(
            self.base_dir, "flac", utt_id + ".flac"
        )

        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.squeeze(0)

        return waveform, label

In [11]:
from scipy.fftpack import dct

def extract_cqcc(waveform, n_coeffs=40):
    spectrum = np.abs(np.fft.rfft(waveform))
    log_spec = np.log(spectrum + 1e-6)
    cepstra = dct(log_spec, norm="ortho")
    return torch.tensor(cepstra[:n_coeffs], dtype=torch.float)

In [12]:
def build_cqcc_cache(dataset, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    for i in range(len(dataset)):
        audio, _ = dataset[i]
        cqcc = extract_cqcc(audio.numpy())
        torch.save(cqcc, f"{save_dir}/{i}.pt")

In [13]:
class ASVspoofWithCQCC(Dataset):
    def __init__(self, audio_dataset, cqcc_dir):
        self.audio_dataset = audio_dataset
        self.cqcc_dir = cqcc_dir

    def __len__(self):
        return len(self.audio_dataset)

    def __getitem__(self, idx):
        audio, label = self.audio_dataset[idx]
        cqcc = torch.load(f"{self.cqcc_dir}/{idx}.pt")
        return audio, cqcc, label

In [14]:
class Wav2VecEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-xls-r-300m"
        )
        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.model(x).last_hidden_state

In [15]:
class XLSRClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Wav2VecEncoder()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(1024, 1)

    def forward(self, x):
        x = self.encoder(x)          # (B, T, 1024)
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)
        return torch.sigmoid(self.fc(x))

In [16]:
class CQCCClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(40, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return torch.sigmoid(self.net(x))

In [17]:
class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.stream_a = XLSRClassifier()
        self.stream_b = CQCCClassifier()

    def forward(self, audio, cqcc):
        p_a = self.stream_a(audio)
        p_b = self.stream_b(cqcc)
        return (p_a + p_b) / 2

In [25]:
print(os.path.exists(TRAIN_PROTOCOL))
print(os.path.exists(DEV_PROTOCOL))
print(len(os.listdir(f"{TRAIN_DIR}/flac")))

True
True
25380


In [26]:
train_audio_ds = ASVspoof2019Dataset(TRAIN_DIR, TRAIN_PROTOCOL)
dev_audio_ds   = ASVspoof2019Dataset(DEV_DIR, DEV_PROTOCOL)

In [27]:
# build_cqcc_cache(train_audio_ds, "/kaggle/working/cqcc_train")
# build_cqcc_cache(dev_audio_ds, "/kaggle/working/cqcc_dev")

In [28]:
len(os.listdir("/kaggle/working/cqcc_train"))

25380

In [29]:
print("Train CQCC files:", len(os.listdir("/kaggle/working/cqcc_train")))
print("Dev CQCC files:", len(os.listdir("/kaggle/working/cqcc_dev")))
print("Train samples:", len(train_audio_ds))
print("Dev samples:", len(dev_audio_ds))

Train CQCC files: 25380
Dev CQCC files: 24844
Train samples: 25380
Dev samples: 24844


In [30]:
train_ds = ASVspoofWithCQCC(train_audio_ds, "/kaggle/working/cqcc_train")
dev_ds   = ASVspoofWithCQCC(dev_audio_ds, "/kaggle/working/cqcc_dev")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn
)

In [34]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device", device)

model = DualStreamModel().to(device)

criterion = nn.BCELoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4
)

using device cuda


In [38]:
EPOCHS = 3

In [ ]:
from tqdm.auto import tqdm 

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=True
    )

    for step, (audio, cqcc, label) in enumerate(pbar):
        audio = audio.to(device)
        cqcc = cqcc.to(device)
        label = label.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        out = model(audio, cqcc)
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        avg_loss = total_loss / (step + 1)

        # Update tqdm bar
        pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{avg_loss:.4f}"
        })

    print(
        f"Epoch {epoch+1}/{EPOCHS} finished | "
        f"Avg Loss: {total_loss/len(train_loader):.4f}"
    )

Epoch 1/3:   0%|          | 0/6345 [00:00<?, ?it/s]

In [ ]:
model.eval()
scores, labels = [], []

with torch.no_grad():
    for audio, cqcc, label in dev_loader:
        audio = audio.to(device)
        cqcc = cqcc.to(device)
        out = model(audio, cqcc)

        scores.extend(out.cpu().numpy())
        labels.extend(label.numpy())

eer = compute_eer(labels, scores)
print("DEV EER:", eer)

In [ ]:
torch.save(model.state_dict(), "dual_stream_baseline.pt")